# 03 — MRR Analysis & Decomposition

**Monthly Recurring Revenue (MRR)** is the north star metric for any subscription business. Unlike one-off revenue, MRR is predictable, comparable across periods, and directly tied to the health of the customer base.

More importantly, MRR decomposition lets us answer the question *why* revenue changed month-over-month by breaking it into its constituent movements:
- **New MRR** — revenue from brand-new customers
- **Expansion MRR** — revenue gained when existing customers upgrade
- **Contraction MRR** — revenue lost when customers downgrade
- **Churned MRR** — revenue lost from customers who cancelled

This four-part view is sometimes called the **MRR waterfall** or **Net Revenue Retention** framework. A business where expansion + new MRR consistently outpaces churn + contraction MRR is said to have 'negative net churn' — the gold standard of SaaS financial health.

## 1. Setup & Load Data

In [ ]:
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../')

from src.mrr import (
    build_monthly_mrr,
    aggregate_mrr,
    build_mrr_decomposition,
    plot_mrr_trend,
    plot_mrr_decomposition,
    plot_churn_rate_trend
)

df = pd.read_csv('../data/processed/customers_processed.csv')
print(f'Loaded: {df.shape[0]:,} customers, {df.shape[1]} columns')
df.head(3)

## 2. Current MRR Snapshot

Before we look at trends, let's establish the current state: total MRR, active customer count, and ARPU (Average Revenue Per User). These three numbers form the fundamental MRR identity:

> **MRR = Active Customers × ARPU**

In [ ]:
# Identify active customers — those who haven't churned
churn_col = 'Churn' if 'Churn' in df.columns else 'churn'
charges_col = 'MonthlyCharges' if 'MonthlyCharges' in df.columns else 'monthly_charges'
contract_col = 'Contract' if 'Contract' in df.columns else 'contract'

# Handle both string and numeric churn encoding
if df[churn_col].dtype == object:
    active_mask = df[churn_col] == 'No'
else:
    active_mask = df[churn_col] == 0

df_active = df[active_mask]

total_mrr = df_active[charges_col].sum()
active_customers = len(df_active)
arpu = total_mrr / active_customers if active_customers > 0 else 0

print('=== Current MRR Snapshot ===')
print(f'  Total MRR:        ${total_mrr:>12,.2f}')
print(f'  Active Customers: {active_customers:>12,}')
print(f'  ARPU:             ${arpu:>12,.2f} / month')

## 3. MRR by Contract Type

Segmenting MRR by contract type reveals both the revenue contribution of each segment and the risk profile. Month-to-month contracts are high-volume but high-churn; annual contracts represent more stable, predictable revenue.

In [ ]:
mrr_by_contract = (
    df_active.groupby(contract_col)[charges_col]
    .agg(['sum', 'count', 'mean'])
    .reset_index()
    .rename(columns={'sum': 'total_mrr', 'count': 'customers', 'mean': 'arpu'})
    .sort_values('total_mrr', ascending=False)
)
mrr_by_contract['mrr_share_pct'] = (mrr_by_contract['total_mrr'] / mrr_by_contract['total_mrr'].sum() * 100).round(1)

print('MRR Breakdown by Contract Type:')
display(mrr_by_contract.style.format({
    'total_mrr': '${:,.0f}',
    'customers': '{:,}',
    'arpu': '${:.2f}',
    'mrr_share_pct': '{:.1f}%'
}))

fig = px.bar(
    mrr_by_contract,
    x=contract_col,
    y='total_mrr',
    color=contract_col,
    text='mrr_share_pct',
    title='MRR Contribution by Contract Type',
    labels={contract_col: 'Contract Type', 'total_mrr': 'Monthly Revenue ($)'}
)
fig.update_traces(texttemplate='%{text:.1f}% of MRR', textposition='outside')
fig.update_layout(plot_bgcolor='white', showlegend=False, height=400)
fig.show()

## 4. Build Monthly MRR Time Series

`build_monthly_mrr()` expands the customer-level snapshot into a longitudinal time series by inferring each customer's active months from their tenure. This enables us to track how MRR evolved historically.

Note: this function processes ~7,000 customers across multiple months, so it may take 15–30 seconds to complete.

In [ ]:
print('Building monthly MRR time series (this may take a moment)...')
long_df = build_monthly_mrr(df)
print(f'Long-format MRR table: {long_df.shape[0]:,} rows')
long_df.head(5)

## 5. Aggregate Monthly MRR

`aggregate_mrr()` collapses the long-format customer-month table into a monthly summary: total active MRR, customer count, ARPU, and churn rate for each calendar month.

In [ ]:
monthly = aggregate_mrr(long_df)
print(f'Monthly MRR summary: {len(monthly)} months of data')
monthly.head(8)

## 6. MRR Trend

A simple time series of total MRR over the observation window. We're looking for consistent upward slope — and specifically whether the slope is accelerating (growth compounding) or decelerating (growth headwinds emerging).

In [ ]:
fig_mrr = plot_mrr_trend(monthly)
fig_mrr.show()

## 7. MRR Decomposition (Waterfall)

Now for the most analytically powerful view: the MRR waterfall. `build_mrr_decomposition()` calculates the monthly movement in each MRR component, so we can see not just *that* MRR changed but *why*.

A healthy decomposition shows New + Expansion MRR consistently exceeding Churned + Contraction MRR, resulting in positive Net New MRR month over month.

In [ ]:
decomp = build_mrr_decomposition(monthly)
print('MRR Decomposition table (first 6 months):')
decomp.head(6)

In [ ]:
fig_decomp = plot_mrr_decomposition(decomp)
fig_decomp.show()

## 8. Churn Rate Trend

The MRR churn rate trend is often more informative than the absolute MRR trend for diagnosing product health. A rising MRR alongside a rising churn rate is a warning sign — the business is running faster on the acquisition treadmill to offset accelerating losses.

In [ ]:
fig_churn_trend = plot_churn_rate_trend(monthly)
fig_churn_trend.show()

## Key Insights

1. **Month-to-month contracts dominate MRR volume but concentrate churn risk.** A significant share of total revenue sits in the highest-churn contract tier. Converting even 10% of month-to-month customers to annual contracts would meaningfully improve revenue predictability.

2. **The MRR decomposition reveals whether growth is 'real'.** If Churned MRR is growing faster than New MRR, the business is acquiring customers at ever-increasing cost simply to stay flat — a sign that unit economics are deteriorating even if the headline MRR chart still looks positive.

3. **ARPU trends deserve close attention.** If ARPU is declining over time, it may indicate that the customer acquisition mix is shifting toward lower-value segments, or that pricing pressure is forcing discounts.

4. **Monthly churn rate is a lagging indicator.** By the time a spike appears in the churn rate trend, the underlying cause (a product issue, a competitive threat, a support failure) has already happened. This reinforces the need for leading indicators — like engagement scores or NPS — that can surface problems before they manifest in revenue figures.